In [0]:
try:
    API_KEY = dbutils.secrets.get("petropulse", "eia_api_key")
except Exception:
    dbutils.widgets.text("eia_key", "")
    API_KEY = dbutils.widgets.get("eia_key")   # paste key into the widget box at the top
assert API_KEY, "No key found"

In [0]:
import requests
from pyspark.sql import functions as F

def eia_get(path, params):
    p = {"api_key": API_KEY, "data[0]": "value", "offset": 0, "length": 5000}
    p.update(params)
    r = requests.get(f"https://api.eia.gov/v2/{path}/data/", params=p, timeout=60)
    r.raise_for_status()
    return r.json()["response"]["data"]

price_rows = eia_get("petroleum/pri/spt", {
    "frequency": "daily",
    "facets[series][]": ["RWTC", "RBRTE"],
    "sort[0][column]": "period", "sort[0][direction]": "desc",
    "length": 200,
})
print("Price rows:", len(price_rows))

price_schema = "period string, series string, value double, units string"
prices = [(r["period"], r["series"], float(r["value"]), r.get("units")) for r in price_rows if r.get("value") is not None]
(spark.createDataFrame(prices, price_schema)
   .withColumn("ingested_at", F.current_timestamp())
   .write.mode("append").saveAsTable("workspace.petropulse_bronze.eia_prices_raw"))

In [0]:
prod_rows = eia_get("petroleum/crd/crpdn", {
    "frequency": "monthly",
    "sort[0][column]": "period", "sort[0][direction]": "desc",
    "length": 500,
})
print("Production rows:", len(prod_rows))
print(prod_rows[:2])   # so we can see the exact columns

In [0]:
%sql
SELECT series, COUNT(*) AS n, MIN(period) AS first_day, MAX(period) AS last_day
FROM workspace.petropulse_bronze.eia_prices_raw
GROUP BY series;

In [0]:
from pyspark.sql import functions as F

rows, offset = [], 0
while True:
    batch = eia_get("petroleum/crd/crpdn", {
        "frequency": "monthly",
        "start": "2025-01",
        "sort[0][column]": "period", "sort[0][direction]": "desc",
        "offset": offset, "length": 5000,
    })
    rows += batch
    if len(batch) < 5000:
        break
    offset += 5000
print("Production rows fetched:", len(rows))

data = [(r["period"], r["duoarea"], r["area-name"], r["product"], r["process"],
         r["process-name"], r["series"], r["series-description"],
         float(r["value"]) if r.get("value") not in (None, "") else None,
         r.get("units")) for r in rows]

schema = """period string, duoarea string, area_name string, product string,
            process string, process_name string, series string,
            series_description string, value double, units string"""

(spark.createDataFrame(data, schema)
   .withColumn("ingested_at", F.current_timestamp())
   .write.mode("append")
   .saveAsTable("workspace.petropulse_bronze.eia_production_raw"))
print("Loaded.")

In [0]:
%sql
SELECT COUNT(*) AS total_rows, MIN(period) AS first_month, MAX(period) AS last_month,
       COUNT(DISTINCT duoarea) AS areas
FROM workspace.petropulse_bronze.eia_production_raw;

In [0]:
%sql
SELECT duoarea, area_name, process_name, units, COUNT(*) AS n
FROM workspace.petropulse_bronze.eia_production_raw
WHERE units = 'MBBL/D' AND process = 'FPF'
GROUP BY duoarea, area_name, process_name, units
ORDER BY duoarea;